In [1]:
# ==========================================
# IMPORT
# ==========================================
import os
import re
import time
import random
import pandas as pd
import cloudscraper

from bs4 import BeautifulSoup

In [2]:
# ==========================================
# MASTER SCHEMA
# ==========================================

ALL_COLUMNS = [
    "series_id",
    "title",
    "url",

    "type",
    "type_origin",
    "genres",
    "tags",
    "language",

    "authors",
    "artists",
    "original_publisher",
    "english_publisher",

    "year",
    "status_coo",
    "chapter_count",
    "is_completed",

    "licensed",
    "completely_translated",
    "release_frequency",

    "rating_score",
    "rating_votes",

    "activity_week_rank",
    "activity_month_rank",
    "activity_all_rank",

    "reading_list_count",
    "reading_list_month_rank",
    "reading_list_all_rank"
]

In [3]:
# ==========================================
# SESSION
# ==========================================

session = cloudscraper.create_scraper(
    browser={
        "browser": "chrome",
        "platform": "windows",
        "desktop": True
    }
)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (X11; Linux x86_64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
]


In [4]:
# ==========================================
# UTILITIES
# ==========================================

def clean_text(text):
    if not text:
        return None

    text = re.sub(r"\s+", " ", text)
    text = text.replace("<<less", "")
    text = text.replace("more>>", "")
    return text.strip()


def safe_request(url, max_try=5):

    for attempt in range(max_try):

        try:

            headers = {
                "User-Agent": random.choice(USER_AGENTS)
            }

            response = session.get(
                url,
                headers=headers,
                timeout=20
            )

            if response.status_code == 200:
                return response

            print(
                f"HTTP {response.status_code} | attempt {attempt+1}"
            )

        except Exception as e:
            print(f"Request Error: {e}")

        sleep_time = random.uniform(5, 10) + (attempt * 3)

        print(
            f"Sleeping {sleep_time:.1f}s..."
        )

        time.sleep(sleep_time)

    return None


def get_list(soup, div_id):

    tags = soup.select(f"div#{div_id} a")

    if tags:
        return ", ".join(
            x.get_text(strip=True)
            for x in tags
        )

    na_tag = soup.select_one(
        f"div#{div_id} .seriesna"
    )

    if na_tag:
        return clean_text(na_tag.get_text())

    return None


def get_text_id(soup, div_id):

    tag = soup.find("div", id=div_id)

    if not tag:
        return None

    return clean_text(
        tag.get_text(" ", strip=True)
    )

In [5]:
# ==========================================
# DETAIL EXTRACTION
# ==========================================

def extract_detailed_attributes(
    soup,
    html_text,
    url
):

    details = {}

    # ==========================
    # TITLE
    # ==========================

    title_tag = soup.select_one(
        "div.seriestitlenu"
    )

    details["title"] = (
        title_tag.get_text(strip=True)
        if title_tag
        else None
    )

    # ==========================
    # SERIES ID
    # ==========================

    pid_match = re.search(
        r"pid=(\d+)",
        html_text
    )

    details["series_id"] = (
        pid_match.group(1)
        if pid_match
        else None
    )

    # ==========================
    # TYPE
    # ==========================

    details["type"] = get_list(
        soup,
        "showtype"
    )

    origin_tag = soup.select_one(
        "#showtype span"
    )

    details["type_origin"] = (
        origin_tag.get_text(strip=True)
        .replace("(", "")
        .replace(")", "")
        if origin_tag
        else None
    )

    # ==========================
    # GENRES / TAGS
    # ==========================

    details["genres"] = get_list(
        soup,
        "seriesgenre"
    )

    details["tags"] = get_list(
        soup,
        "showtags"
    )

    details["language"] = get_list(
        soup,
        "showlang"
    )

    # ==========================
    # AUTHORS
    # ==========================

    details["authors"] = get_list(
        soup,
        "showauthors"
    )

    details["artists"] = get_list(
        soup,
        "showartists"
    )

    details["original_publisher"] = get_list(
        soup,
        "showopublisher"
    )

    details["english_publisher"] = get_list(
        soup,
        "showepublisher"
    )

    # ==========================
    # STATUS
    # ==========================

    details["year"] = get_text_id(
        soup,
        "edityear"
    )

    status_text = get_text_id(
        soup,
        "editstatus"
    )

    details["status_coo"] = status_text

    chapter_match = re.search(
        r"(\d+)\s*chapters?",
        status_text or "",
        re.IGNORECASE
    )

    if not chapter_match:

        chapter_match = re.search(
            r"(\d+)",
            status_text or ""
        )

    details["chapter_count"] = (
        chapter_match.group(1)
        if chapter_match
        else None
    )

    details["is_completed"] = bool(
        re.search(
            r"completed",
            status_text or "",
            re.IGNORECASE
        )
    )

    details["licensed"] = get_text_id(
        soup,
        "showlicensed"
    )

    details["completely_translated"] = get_text_id(
        soup,
        "showtranslated"
    )

    # ==========================
    # RATING
    # ==========================

    rating_tag = soup.select_one(
        "span.uvotes"
    )

    if rating_tag:

        text = rating_tag.get_text(
            strip=True
        )

        score = re.search(
            r"\(([\d.]+)\s*/",
            text
        )

        votes = re.search(
            r"([\d,]+)\s*votes",
            text
        )

        details["rating_score"] = (
            score.group(1)
            if score
            else None
        )

        details["rating_votes"] = (
            votes.group(1).replace(",", "")
            if votes
            else None
        )

    else:

        details["rating_score"] = None
        details["rating_votes"] = None

    # ==========================
    # RELEASE FREQUENCY
    # ==========================

    freq_match = re.search(
        r"Release Frequency.*?</h5>\s*([^<]+)",
        html_text,
        re.DOTALL
    )

    details["release_frequency"] = (
        clean_text(freq_match.group(1))
        if freq_match
        else None
    )

    # ==========================================
    # ACTIVITY STATS
    # ==========================================

    week = re.search(
        r'Weekly Rank:\s*<span class="userrate rank">#([\d,]+)</span>',
        html_text,
        re.I | re.S
    )

    month = re.search(
        r'Monthly Rank:\s*<span class="userrate rank">#([\d,]+)</span>',
        html_text,
        re.I | re.S
    )

    alltime = re.search(
        r'All Time Rank:\s*<span class="userrate rank">#([\d,]+)</span>',
        html_text,
        re.I | re.S
    )

    details["activity_week_rank"] = (
        week.group(1).replace(",", "")
        if week else None
    )

    details["activity_month_rank"] = (
        month.group(1).replace(",", "")
        if month else None
    )

    details["activity_all_rank"] = (
        alltime.group(1).replace(",", "")
        if alltime else None
    )

    # ==========================================
    # READING LIST
    # ==========================================

    rlist = soup.select_one("b.rlist")

    details["reading_list_count"] = (
        rlist.get_text(strip=True).replace(",", "")
        if rlist
        else None
    )

    reading_section = re.search(
        r'Reading List.*?On\s*<b class="rlist">.*?</b>.*?'
        r'Monthly Rank:\s*<span class="userrate rank">#([\d,]+)</span>.*?'
        r'All Time Rank:\s*<span class="userrate rank">#([\d,]+)</span>',
        html_text,
        re.I | re.S
    )

    if reading_section:
        details["reading_list_month_rank"] = (
            reading_section.group(1).replace(",", "")
        )

        details["reading_list_all_rank"] = (
            reading_section.group(2).replace(",", "")
        )
    else:
        details["reading_list_month_rank"] = None
        details["reading_list_all_rank"] = None

    return details

In [6]:
# ==========================================
# CLEAN DF
# ==========================================

def clean_dataframe(df):

    numeric_cols = [
        "chapter_count",
        "rating_score",
        "rating_votes",
        "activity_week_rank",
        "activity_month_rank",
        "activity_all_rank",
        "reading_list_count",
        "reading_list_month_rank",
        "reading_list_all_rank"
    ]

    for col in numeric_cols:

        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    for col in df.columns:

        if col not in numeric_cols:
            df[col] = df[col].fillna(
                "N/A"
            )

    return df


# ==========================================
# MAIN SCRAPER
# ==========================================
def scrape_novel_details(
    input_csv,
    final_csv,
    save_every=5
):
    # 1. Baca CSV dengan deteksi koma atau titik koma otomatis
    try:
        df = pd.read_csv(input_csv, sep=';')
        if len(df.columns) <= 1:
            df = pd.read_csv(input_csv, sep=',')
    except:
        df = pd.read_csv(input_csv)

    # 2. Sinkronisasi kolom
    for col in ALL_COLUMNS:
        if col not in df.columns:
            df[col] = None

    df = df[
        [c for c in df.columns if c in ALL_COLUMNS]
    ]

    total = len(df)
    df = df.astype(object)
    for idx in df.index:
        row = df.loc[idx]

        # ==========================================
        # INI KUNCI AGAR DIA MELANJUTKAN YANG BELUM
        # ==========================================
        if pd.notna(row.get('authors')) and row.get('authors') != 'N/A' and row.get('authors') != 0:
            continue

        url = row["url"]

        if (
            not isinstance(url, str)
            or not url.startswith("http")
        ):
            continue

        print(f"[{idx+1}/{total}] {row['title']}")

        response = safe_request(url)

        if response:
            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            # Pastikan argumen pemanggilan fungsi ini sesuai dengan yang kamu definisikan
            attrs = extract_detailed_attributes(
                soup,
                response.text,
                url
            )

            for k, v in attrs.items():
                df.at[idx, k] = v

        else:
            print("FAILED")

        time.sleep(
            random.uniform(3, 7)
        )

        if (idx + 1) % save_every == 0:
            temp = clean_dataframe(
                df.copy()
            )

            # Tambahkan quoting=csv.QUOTE_ALL jika import csv sudah ditambahkan
            import csv
            temp.to_csv(
                final_csv,
                index=False,
                encoding="utf-8-sig",
                quoting=csv.QUOTE_ALL
            )

            print(
                f"Checkpoint saved ({idx+1})"
            )

    df = clean_dataframe(df)

    import csv
    df.to_csv(
        final_csv,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_ALL
    )

    print(
        "\nSUCCESS! Scraping completed."
    )


In [7]:
# ==========================================
# EXECUTE
# ==========================================

INPUT_CSV = "./dataset/novel_dataset_9.csv"

FINAL_CSV = (
    "./dataset/"
    "novel_dataset_enriched_FINAL.csv"
)

print(
    "STARTING DETAIL SCRAPER..."
)

if os.path.exists(FINAL_CSV):

    scrape_novel_details(
        FINAL_CSV,
        FINAL_CSV,
        save_every=5
    )

else:

    scrape_novel_details(
        INPUT_CSV,
        FINAL_CSV,

        save_every=5
    )

print("\nPIPELINE FINISHED.")

STARTING DETAIL SCRAPER...
[107/5000] The Kansai Villain’s Rebirth: I Accidentally Ended Up With a Harem
[205/5000] Transmigrated as the Male Lead’s Vicious First Love and Kissed Until Crying
Checkpoint saved (205)
[352/5000] Help! I’ve Been Controlled By An Evil God Player!
[402/5000] The Merman I Picked Up Is Too Delicious
[425/5000] I Deliver Takeout Across the Multiverse
Checkpoint saved (425)
[502/5000] Immortality Begins with Comprehending the Dao Scripture
[522/5000] I’m Just a Death Warrior, Why Must the Immortal Maidens Die for Me?
[538/5000] Falling in Love with a 500-year-old female elder
[546/5000] Peppermint Addiction
[556/5000] The Little Female Is Enchanting And Seductive; Her Many Beast-Husbands Lose Control And Fall For Her
[565/5000] Transmigrated as the Only Ordinary Girl in an All-Alpha Military Academy
Checkpoint saved (565)
[585/5000] A Sword Comes: I – Chen Ping’an Have Only One Sword
Checkpoint saved (585)
[603/5000] Inventory of HSR: Whose Characters Will Break